# CME Futures: Double Machine Learning

This notebook estimates the configured treatment effect for each return horizon. The request
resolves the outcome, treatment, confounders, timing, embargo, nuisance models, and placebo design
before fitting. Missing confounders are an error rather than an invitation to change the estimand.

Nuisance models train on complete timestamp panels before the holdout. HAC uncertainty uses the
declared temporal ordering, and contiguous within-product blocks define the placebo refutation.
`12_model_analysis` reports the causal diagnostics. Trading configuration selection uses
validation backtest Sharpe.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures double-machine-learning requests."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    open_study,
    product_universe_table,
)
from case_studies.research import supersedes_for

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
SUPERSEDES_CAUSAL: str = ""

## Resolve the estimands

Preview row and fold limits must be named in `PREVIEW_REDUCTIONS`. They enter the computation
identity and cannot be mistaken for canonical causal estimates.

In [4]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "preview" and (WORKSPACE is None or not PREVIEW_REDUCTIONS):
    raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [5]:
requests = tuple(
    study.causal(
        method="dml",
        label=label,
        execution_tier=EXECUTION_TIER,
        preview_reductions=PREVIEW_REDUCTIONS,
        supersedes=supersedes_for(SUPERSEDES_CAUSAL, label, labels=list(ALL_LABELS)),
    ).resolve()
    for label in ALL_LABELS
)
request_rows = []
for request in requests:
    computation = request.spec["computation"]
    estimand = computation["estimand"]
    population = computation["analysis_population"]
    request_rows.append(
        {
            "label": request.spec["label"],
            "treatment": estimand["treatment"],
            "outcome": estimand["outcome"],
            "outcome_horizon": estimand["outcome_horizon"],
            "confounders": ", ".join(estimand["confounders"]),
            "analysis_rows": population["n_rows"],
            "analysis_timestamps": population["n_timestamps"],
            "request_hash": request.identity,
        }
    )
request_table = pl.DataFrame(request_rows)
request_table

label,treatment,outcome,outcome_horizon,confounders,analysis_rows,analysis_timestamps,request_hash
str,str,str,str,str,i64,i64,str
"""fwd_ret_5d""","""carry_pct""","""fwd_ret_5d""","""5 days 00:00:00""","""vol_21d, momentum_composite, c…",88633,3098,"""1ec57a7299b0"""
"""fwd_ret_21d""","""carry_pct""","""fwd_ret_21d""","""21 days 00:00:00""","""vol_21d, momentum_composite, c…",88105,3082,"""e47b499b50ef"""


## Execute and verify restart

The shared causal runner uses deterministic seeds and persists one immutable result per resolved
request. Reopening the same request must return the same identity and a complete result.

In [6]:
results = []
for request in requests:
    label = request.spec["label"]
    result = request.run()
    restarted = request.run()
    if not result.complete or restarted.hash != result.hash:
        raise RuntimeError(f"causal request for {label} did not persist completely")
    results.append(
        {
            "label": label,
            "execution_tier": result.execution_tier,
            "complete": result.complete,
            "causal_hash": result.hash,
        }
    )

~/ml4t/public-s6-cme_futures/case_studies/utils/causal.py:1566: UserWarning: block permutation with block_size=21 cannot move 0.9% of the treatment rows: they sit in segments too short to hold two blocks, so the placebo distribution holds them at their observed values and the refutation p-value is biased toward 1. Read placebo_frozen_fraction alongside the p-value, and lower block_size or widen gap_tolerance if the frozen share is large.
  results = run_dml_analysis(


In [7]:
pl.DataFrame(results).sort("label")

label,execution_tier,complete,causal_hash
str,str,bool,str
"""fwd_ret_21d""","""canonical""",true,"""e47b499b50ef"""
"""fwd_ret_5d""","""canonical""",true,"""1ec57a7299b0"""
